In [1]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='gee-space-hack')

In [2]:
# ======================== CONFIGURACION ========================

AOI = ee.FeatureCollection('projects/gee-space-hack/assets/SpaceHack/AOI_GreaterGuayaquil_v2')
ASSET_ROOT = 'projects/gee-space-hack/assets/SpaceHack'

YEARS = [2022, 2023, 2024,2025]
MONTH_START = 3
MONTH_END = 12

CLOUD_FILTER = 100
CLD_PRB_THRESH = 30
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 40
SR_BAND_SCALE = 1e4

In [3]:
# ======================== FUNCIONES BASE ========================

def s2_cld(aoi, start_date, end_date):
    s2_sr_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER)))

    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(aoi)
        .filterDate(start_date, end_date))

    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))


def add_mndwi(img):
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return img.addBands(mndwi)


def add_cloud_bands(img):
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')
    return img.addBands(ee.Image([cld_prb, is_cloud]))


def add_shadow_bands(img):
    not_water = img.select('MNDWI').gt(0.1)
    dark_pixels = (img.select('B8')
        .lt(NIR_DRK_THRESH * SR_BAND_SCALE)
        .multiply(not_water)
        .rename('dark_pixels'))

    shadow_azimuth = ee.Number(90).subtract(
        ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))

    cld_proj = (img.select('clouds')
        .directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST * 10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    shadows = cld_proj.multiply(dark_pixels).rename('shadows')
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))


def add_cld_shdw_mask(img):
    img_cloud = add_cloud_bands(img)
    img_cloud_shadow = add_shadow_bands(img_cloud)
    is_cld_shdw = (img_cloud_shadow.select('clouds')
        .add(img_cloud_shadow.select('shadows')).gt(0))
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER * 2 / 20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))
    return img_cloud_shadow.addBands(is_cld_shdw)


def apply_cld_shdw_mask(img):
    not_cld_shdw = img.select('cloudmask').Not()
    return img.select('B.*').updateMask(not_cld_shdw)


In [4]:
# ======================== INDICES DE VEGETACION ========================

def add_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = img.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndmi = img.normalizedDifference(['B11', 'B3']).rename('NDMI')
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    ndre = img.normalizedDifference(['B8', 'B5']).rename('NDRE')
    ndbi = img.normalizedDifference(['B11', 'B8']).rename('NDBI')

    savi = img.expression(
        '((NIR - RED) / (NIR + RED + L)) * (1 + L)',
        {'NIR': img.select('B8'), 'RED': img.select('B4'), 'L': 0.5}
    ).rename('SAVI')

    mmri = img.expression(
        '(abs((GREEN - SWIR1) / (GREEN + SWIR1)) - abs((NIR - RED) / (NIR + RED))) / '
        '(abs((GREEN - SWIR1) / (GREEN + SWIR1)) + abs((NIR - RED) / (NIR + RED)))',
        {'GREEN': img.select('B3'), 'SWIR1': img.select('B11'),
         'NIR': img.select('B8'), 'RED': img.select('B4')}
    ).rename('MMRI')

    mvi = img.expression(
        '(NIR - GREEN) / (SWIR2 - GREEN)',
        {'NIR': img.select('B8'), 'GREEN': img.select('B3'),
         'SWIR2': img.select('B12')}
    ).rename('MVI')

    evi = img.expression(
        '2.5 * (NIR - RED) / (NIR + 2.4 * RED + 1)',
        {'NIR': img.select('B8'), 'RED': img.select('B4')}
    ).rename('EVI')

    return (img.addBands(ndvi).addBands(ndwi).addBands(ndmi)
        .addBands(mndwi).addBands(ndre).addBands(savi)
        .addBands(mmri).addBands(mvi).addBands(evi))



In [5]:
# ======================== PIPELINE PRINCIPAL ========================

def generate_composite(aoi, year, month_start, month_end):
    start_date = ee.Date.fromYMD(year, month_start, 1)
    end_date = ee.Date.fromYMD(year, month_end, 31)

    collection = s2_cld(aoi, start_date, end_date)
    n_images = collection.size()

    processed = (collection
        .map(add_mndwi)
        .map(add_cld_shdw_mask)
        .map(apply_cld_shdw_mask)
        .map(add_indices))

    composite = processed.median()
    return composite, n_images


def export_composite(composite, aoi, year, scale=10, fast=False):
    suffix = '_fast' if fast else ''
    region = aoi.geometry().centroid().buffer(5000) if fast else aoi.geometry()
    export_scale = 20 if fast else scale

    task = ee.batch.Export.image.toAsset(**{
        'image': composite,
        'description': f'COM_{year}{suffix}',
        'assetId': f'{ASSET_ROOT}/COM_{year}{suffix}',
        'scale': export_scale,
        'region': region,
        'maxPixels': 1e10
    })
    task.start()
    return task

In [6]:
# ======================== EJECUCION ========================

tasks = []

for year in YEARS:
    composite, n_images = generate_composite(AOI, year, MONTH_START, MONTH_END)

    print(f'--- {year} ---')
    print(f'Imagenes disponibles: {n_images.getInfo()}')

    task_fast = export_composite(composite, AOI, year, fast=True)
    tasks.append((year, 'fast', task_fast))
    print(f'Export rapido lanzado: COM_{year}_fast')

    task_full = export_composite(composite, AOI, year, scale=10, fast=False)
    tasks.append((year, 'full', task_full))
    print(f'Export completo lanzado: COM_{year}')

print('\n=== Todos los exports lanzados ===')
print('Usa ee.batch.Task.list() para monitorear el progreso.')

--- 2022 ---
Imagenes disponibles: 672
Export rapido lanzado: COM_2022_fast
Export completo lanzado: COM_2022
--- 2023 ---
Imagenes disponibles: 673
Export rapido lanzado: COM_2023_fast
Export completo lanzado: COM_2023
--- 2024 ---
Imagenes disponibles: 681
Export rapido lanzado: COM_2024_fast
Export completo lanzado: COM_2024
--- 2025 ---
Imagenes disponibles: 922
Export rapido lanzado: COM_2025_fast
Export completo lanzado: COM_2025

=== Todos los exports lanzados ===
Usa ee.batch.Task.list() para monitorear el progreso.


In [7]:
# ======================== MONITOREO ========================

import time

def check_tasks(tasks, interval=60):
    """Monitorea el estado de los exports. Ejecutar en celda separada."""
    pending = list(tasks)
    while pending:
        still_pending = []
        for year, mode, task in pending:
            status = task.status()
            state = status['state']
            if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
                print(f'[{state}] COM_{year}_{mode}')
                if state == 'FAILED':
                    print(f'  Error: {status.get("error_message", "desconocido")}')
            else:
                still_pending.append((year, mode, task))
        pending = still_pending
        if pending:
            print(f'Pendientes: {len(pending)} tareas. Revisando en {interval}s...')
            time.sleep(interval)
    print('Todos los exports finalizados.')
